# Procurement Cycle-Time Analysis


## Project Overview

This notebook analyses 1 200 purchase requests to measure and decompose procurement cycle time across approval, PO issuance, and delivery phases.


## Learning Objectives

- Decompose total cycle time into sub-phases
- Identify departments and vendors causing approval delays
- Segment cycle time by order value using Pareto analysis
- Visualise delay drivers and make data-driven recommendations


## Business or Research Problem

Long procurement cycle times inflate working capital requirements and disrupt production schedules. Understanding where time is lost enables targeted process improvements.


## Why This Analysis Matters

Even a 20% reduction in total cycle time can meaningfully reduce inventory holding requirements and improve vendor relationships.


## Dataset Overview

- **Rows:** 1 200 purchase requests
- **Columns:** pr_id, department, category, pr_date, approval_date, po_date, delivery_date, value, vendor_id
- **Derived:** approval_days, po_days, delivery_days, total_cycle_days


## Dataset Source and License Notes

Fully synthetic data, NumPy seed=42. Educational use only.


## Environment Setup


In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Python:', sys.version)


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Configuration / Constants


In [ ]:
SEED = 42
N = 1200
DEPARTMENTS = ['Engineering', 'Operations', 'IT', 'Finance', 'HR', 'Facilities']
CATEGORIES = ['Equipment', 'Software', 'Consumables', 'Services', 'MRO']
N_VENDORS = 30


## Dataset Download or Loading


In [ ]:
rng = np.random.default_rng(SEED)
dept_delay = {'Engineering': 3, 'Operations': 2, 'IT': 5, 'Finance': 4, 'HR': 2, 'Facilities': 3}

departments = rng.choice(DEPARTMENTS, size=N)
categories = rng.choice(CATEGORIES, size=N)
vendor_ids = [f'V{str(i).zfill(3)}' for i in rng.integers(1, N_VENDORS+1, N)]
values = rng.lognormal(mean=8, sigma=1.5, size=N).round(2)

pr_dates = pd.to_datetime('2022-01-01') + pd.to_timedelta(rng.integers(0, 700, N), unit='D')
approval_days = np.array([max(1, int(rng.normal(dept_delay[d] + 3, 2))) for d in departments])
po_days = rng.integers(1, 5, N)
delivery_days = rng.integers(5, 30, N)

approval_dates = pr_dates + pd.to_timedelta(approval_days, unit='D')
po_dates = approval_dates + pd.to_timedelta(po_days, unit='D')
delivery_dates = po_dates + pd.to_timedelta(delivery_days, unit='D')
total_cycle_days = approval_days + po_days + delivery_days

df = pd.DataFrame({'pr_id': [f'PR{str(i).zfill(5)}' for i in range(1, N+1)],
    'department': departments, 'category': categories, 'vendor_id': vendor_ids,
    'pr_date': pr_dates, 'approval_date': approval_dates, 'po_date': po_dates,
    'delivery_date': delivery_dates, 'value': values,
    'approval_days': approval_days, 'po_days': po_days,
    'delivery_days': delivery_days, 'total_cycle_days': total_cycle_days})
print(df.shape)
df.head(3)


## Data Validation Checks


In [ ]:
print('Missing values:\n', df.isnull().sum())
assert (df['approval_days'] > 0).all()
assert (df['delivery_date'] > df['pr_date']).all()
print('All validations passed.')


## Data Cleaning


In [ ]:
df['value'] = df['value'].clip(lower=100)
print('Clean shape:', df.shape)
print('Value range: ${:,.0f} - ${:,.0f}'.format(df['value'].min(), df['value'].max()))


## Exploratory Data Analysis


In [ ]:
print(df[['approval_days','po_days','delivery_days','total_cycle_days']].describe().round(2))
avg_cycle = df['total_cycle_days'].mean()
print(f'\nAverage total cycle time: {avg_cycle:.1f} days')


## Univariate Analysis

Distribution of total cycle time.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['total_cycle_days'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title('Total Cycle Time Distribution')
axes[0].set_xlabel('Days')
axes[1].hist(np.log1p(df['value']), bins=25, color='salmon', edgecolor='white')
axes[1].set_title('Log-Transformed Order Value')
axes[1].set_xlabel('log(1+value)')
plt.tight_layout()
plt.show()


## Bivariate / Multivariate Analysis

Cycle time breakdown by phase and department.


In [ ]:
dept_cycle = df.groupby('department')[['approval_days','po_days','delivery_days']].mean().round(1)
dept_cycle.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='Set2')
plt.title('Average Cycle Time Breakdown by Department')
plt.ylabel('Days')
plt.legend(title='Phase')
plt.tight_layout()
plt.show()
slowest_dept = df.groupby('department')['approval_days'].mean().idxmax()
print(f'Slowest department for approval: {slowest_dept}')


## Feature-Specific Insights

Vendor lead time distribution — top 10 vendors by mean delivery days.


In [ ]:
vendor_lead = df.groupby('vendor_id')['delivery_days'].mean().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 4))
vendor_lead.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Top 10 Vendors by Mean Delivery Days')
ax.set_ylabel('Mean Delivery Days')
plt.tight_layout()
plt.show()


## Statistical Checks

Do high-value orders have longer cycle times than low-value orders?


In [ ]:
median_val = df['value'].median()
high_val = df[df['value'] >= median_val]['total_cycle_days']
low_val = df[df['value'] < median_val]['total_cycle_days']
t_stat, p_val = stats.ttest_ind(high_val, low_val)
print(f'T-test (high vs low value orders): t={t_stat:.2f}, p={p_val:.4f}')
print(f'High-value mean cycle: {high_val.mean():.1f} days | Low-value mean cycle: {low_val.mean():.1f} days')


## Visual Analysis

Pareto of total cycle days — which departments account for 80% of delay?


In [ ]:
dept_total = df.groupby('department')['total_cycle_days'].sum().sort_values(ascending=False)
cumulative = dept_total.cumsum() / dept_total.sum() * 100
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(dept_total.index, dept_total.values, color='steelblue')
ax2.plot(dept_total.index, cumulative.values, marker='o', color='red', label='Cumulative %')
ax2.axhline(80, color='orange', linestyle='--')
ax1.set_title('Pareto: Total Cycle Days by Department')
ax1.set_ylabel('Total Cycle Days')
ax2.set_ylabel('Cumulative %')
plt.tight_layout()
plt.show()


## Key Findings


In [ ]:
print(f'1. Average total procurement cycle time: {avg_cycle:.1f} days')
print(f'2. Slowest approving department: {slowest_dept}')
top_vendor = vendor_lead.index[0]
print(f'3. Slowest vendor (delivery): {top_vendor} with {vendor_lead.iloc[0]:.1f} days avg')
print(f'4. High-value orders cycle time: {high_val.mean():.1f} days vs {low_val.mean():.1f} days for low-value')
print(f'5. T-test p-value: {p_val:.4f}')


## Limitations

- Synthetic data does not model approval thresholds or multi-level approvals
- Vendor lead times are not linked to category or contract terms
- Seasonal or holiday effects on delivery are not simulated


## Recommendations / Next Steps

1. Streamline IT department approval workflow — highest mean approval time
2. Set vendor performance SLAs with escalation triggers beyond N delivery days
3. Fast-track low-value orders (<$1K) with automated approvals
4. Build a monthly cycle time dashboard for procurement leadership


## Common Mistakes

- Averaging cycle time across departments without weighting by volume hides bottlenecks
- Ignoring PO issuance delay (often treated as instant but can be 1-3 days)
- Focusing only on delivery without fixing the upstream approval bottleneck


## Mini Challenge / Exercises

1. Compute the 90th percentile cycle time per department and flag outliers
2. Group by `category` and identify which procurement category has the longest average delivery
3. Scatter plot `value` vs `total_cycle_days` — is there a linear relationship?


## Final Summary / Key Takeaways

- Procurement cycle time decomposes clearly into approval, PO issuance, and delivery phases
- Approval delay is the most controllable sub-phase and the highest-impact improvement area
- Vendor lead times compound approval delays — both must be addressed simultaneously
- Pareto analysis guides where to focus process re-engineering efforts
